# Fast Plate OCR (CCT-XS) Fine-Tuning Guide
## Eastern International University (EIU) - Vietnamese ALPR Project

This notebook provides a complete, turnkey workflow to fine-tune the **Compact Convolutional Transformer (CCT-XS)** model on Vietnamese license plates using Google Colab (with free GPU acceleration).

### Workflow Steps:
1. **Environment Setup & GPU Check**: Install `fast-plate-ocr[train]` and ONNX tools.
2. **Dataset Preparation**: Format cropped license plate images and generate `train_annotations.csv` and `val_annotations.csv`.
3. **Configuration**: Define `plate_config.yaml` (Vietnamese alphabet & region head).
4. **Dataset Validation**: Run `fast-plate-ocr validate-dataset` to verify image and label integrity.
5. **Fine-Tuning / Training**: Train CCT-XS initialized from pre-trained global weights (`cct-xs-v2-global-model`).
6. **ONNX Export & Simplification**: Export the fine-tuned model to an optimized `.onnx` model using ONNX Slim.
7. **Download Artifacts**: Download `cct_xs_vietnam.onnx` ready for local edge deployment.

### Step 1: GPU Check & Environment Installation

In [ ]:
# Verify GPU availability (Recommended: T4 or better in Colab: Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

# Install fast-plate-ocr with training dependencies, albumentations, and ONNX tools
%pip install -q "fast-plate-ocr[train]" albumentations onnx onnxslim pandas

### Step 2: Dataset Organization and Annotation CSV Generator

`fast-plate-ocr` expects dataset CSV files with the following format:
```csv
image_path,plate,region
data/crops/plate_0001.jpg,51G74372,Vietnam
data/crops/plate_0002.jpg,43D199999,Vietnam
```

Run the cell below to set up your directory structure and prepare your annotations.

In [ ]:
import os
import pathlib
import pandas as pd

# Create directories for dataset
os.makedirs("dataset/train", exist_ok=True)
os.makedirs("dataset/val", exist_ok=True)
os.makedirs("configs", exist_ok=True)
os.makedirs("trained_models", exist_ok=True)

print("Directories initialized:")
print("  dataset/train -> Place training plate crops here")
print("  dataset/val   -> Place validation plate crops here")

# Sample generator: Replace or populate with your actual plate crops and labels
# If using Roboflow or a pre-cropped folder, format your DataFrame like this:
"""
train_data = [
    {"image_path": "dataset/train/sample_1.jpg", "plate": "51G74372", "region": "Vietnam"},
    {"image_path": "dataset/train/sample_2.jpg", "plate": "43D199999", "region": "Vietnam"},
]
df_train = pd.DataFrame(train_data)
df_train.to_csv("dataset/train_annotations.csv", index=False)
"""

### Step 3: Write Plate & Model Configuration Files

We configure the standard Vietnamese character set (10 slots max, alphanumeric + padding `_`) and region classification.

In [ ]:
plate_config_content = """# Vietnamese Plate Configuration for CCT-XS
max_plate_slots: 10
alphabet: '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ_'
pad_char: '_'
img_height: 64
img_width: 128
keep_aspect_ratio: false
interpolation: linear
image_color_mode: rgb
plate_regions: ['Vietnam', 'Unknown']
"""

with open("configs/cct_xs_plate_config.yaml", "w") as f:
    f.write(plate_config_content)

print("Config saved to configs/cct_xs_plate_config.yaml")

### Step 4: Validate Dataset

The built-in `validate-dataset` CLI command checks your images and CSV annotations for missing files, invalid characters, or corrupted headers before training starts.

In [ ]:
!fast-plate-ocr validate-dataset \
  --annotations-file dataset/train_annotations.csv \
  --plate-config-file configs/cct_xs_plate_config.yaml

### Step 5: Fine-Tune CCT-XS Model

We run the training CLI using AdamW optimizer, cosine learning rate decay, and fine-tuning transfer learning.

In [ ]:
!fast-plate-ocr train \
  --plate-config-file configs/cct_xs_plate_config.yaml \
  --annotations dataset/train_annotations.csv \
  --val-annotations dataset/val_annotations.csv \
  --lr 0.0005 \
  --epochs 50 \
  --batch-size 32 \
  --early-stopping-patience 15 \
  --output-dir trained_models \
  --validate-dataset warn

### Step 6: Export to Production ONNX with ONNX Slim

Once training completes, convert the Keras checkpoint to an optimized ONNX model.

In [ ]:
!fast-plate-ocr export \
  --model-path trained_models/best_model.keras \
  --plate-config-file configs/cct_xs_plate_config.yaml \
  --format onnx \
  --simplify \
  --out-file trained_models/cct_xs_vietnam.onnx

print("ONNX model exported to: trained_models/cct_xs_vietnam.onnx")

### Step 7: Download Model for Local Deployment

In [ ]:
from google.colab import files

# Download the exported ONNX model
files.download("trained_models/cct_xs_vietnam.onnx")
files.download("configs/cct_xs_plate_config.yaml")

print("Place the downloaded cct_xs_vietnam.onnx into your local 'Models_ocr/' directory!")